In [1]:
import os
import sys
from pathlib import Path

# Remonte à la racine du projet si le notebook est exécuté depuis notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src.collect_licences import get_licences_resources, download_file, RAW_DIR
import pandas as pd

from src.collect_licences import get_licences_resources, download_file, RAW_DIR
import pandas as pd

# 1. Voir tous les fichiers disponibles
df_res = get_licences_resources()
print(df_res[["title", "size_mb"]])

# 2. Télécharger uniquement 2023 pour tester
row_2023 = df_res[df_res["title"].str.contains("2023")].iloc[0]
path_2023 = download_file(row_2023["url"], RAW_DIR / "lic-data-2023.csv")

# 3. Lire uniquement les 5 premières lignes pour vérifier le format
apercu = pd.read_csv(path_2023, sep=";", nrows=5)
print(apercu.columns.tolist())
apercu.head()

               title  size_mb
0  lic-data-2023.csv    256.4
1  lic-data-2022.csv    247.7
2  lic-data-2021.csv    225.8
3  lic-data-2020.csv    240.5
4  lic-data-2019.csv    266.8
5  lic-data-2018.csv    206.3
6  lic-data-2017.csv    222.2
7  lic-data-2016.csv    230.6
Déjà présent : lic-data-2023.csv
['Code Commune', 'Commune', 'Code QPV', 'Nom QPV', 'Département', 'Région', 'Statut géo', 'Code', 'Fédération', 'F - 1 à 4 ans', 'F - 5 à 9 ans', 'F - 10 à 14 ans', 'F - 15 à 19 ans', 'F - 20 à 24 ans', 'F - 25 à 29 ans', 'F - 30 à 34 ans', 'F - 35 à 39 ans', 'F - 40 à 44 ans', 'F - 45 à 49 ans', 'F - 50 à 54 ans', 'F - 55 à 59 ans', 'F - 60 à 64 ans', 'F - 65 à 69 ans', 'F - 70 à 74 ans', 'F - 75 à 79 ans', 'F - 80 à 99 ans', 'F - NR', 'H - 1 à 4 ans', 'H - 5 à 9 ans', 'H - 10 à 14 ans', 'H - 15 à 19 ans', 'H - 20 à 24 ans', 'H - 25 à 29 ans', 'H - 30 à 34 ans', 'H - 35 à 39 ans', 'H - 40 à 44 ans', 'H - 45 à 49 ans', 'H - 50 à 54 ans', 'H - 55 à 59 ans', 'H - 60 à 64 ans', 'H - 65 à 69 

,Code Commune,Commune,Code QPV,Nom QPV,Département,Région,Statut géo,Code,Fédération,F - 1 à 4 ans,...,H - 45 à 49 ans,H - 50 à 54 ans,H - 55 à 59 ans,H - 60 à 64 ans,H - 65 à 69 ans,H - 70 à 74 ans,H - 75 à 79 ans,H - 80 à 99 ans,H - NR,Total
0,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,101,FF d'Athlétisme,0,...,0,0,0,0,0,0,0,0,0,6
1,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,103,FF de Badminton,0,...,0,2,1,0,0,0,0,0,0,11
2,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,105,FF de Basketball,0,...,0,0,0,0,0,0,0,0,0,6
3,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,108,FF de Cyclisme,0,...,0,0,0,0,0,0,0,0,0,2
4,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,109,FF d'Équitation,0,...,0,0,0,0,2,0,0,0,0,11


In [6]:
def aggregate_by_federation(csv_path: Path, group_cols: list[str], value_cols: list[str]) -> pd.DataFrame:
    """Lit un gros CSV par paquets et agrège par fédération pour économiser la RAM."""
    chunks_agg = []
    usecols = group_cols + value_cols

    for chunk in pd.read_csv(csv_path, sep=";", usecols=usecols, chunksize=100_000, low_memory=False):
        agg = chunk.groupby(group_cols, as_index=False)[value_cols].sum()
        chunks_agg.append(agg)

    df_final = pd.concat(chunks_agg, ignore_index=True)
    return df_final.groupby(group_cols, as_index=False)[value_cols].sum()

In [5]:
import os
import sys
from pathlib import Path
import pandas as pd

# Remonte à la racine du projet si le notebook est exécuté depuis notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from src.collect_licences import get_licences_resources, download_file, RAW_DIR


def lire_csv_auto(chemin: Path, **kwargs) -> pd.DataFrame:
    """Lit un CSV en testant UTF-8 puis Latin-1 selon le millésime."""
    for enc in ("utf-8", "latin1"):
        try:
            return pd.read_csv(chemin, sep=";", encoding=enc, **kwargs)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Impossible de décoder le fichier : {chemin}")


# 1. Lister tous les fichiers disponibles
df_res = get_licences_resources()
print(df_res[["title", "size_mb"]])

# 2. Télécharger toutes les années disponibles et vérifier leurs colonnes
fichiers_telecharges = {}

for _, row in df_res.iterrows():
    nom_fichier = row["title"] if row["title"].endswith(".csv") else f"{row['title']}.csv"
    chemin_local = download_file(row["url"], RAW_DIR / nom_fichier)
    fichiers_telecharges[row["title"]] = chemin_local

    # Aperçu rapide des 3 premières lignes avec gestion automatique de l'encodage
    apercu = lire_csv_auto(chemin_local, nrows=3)
    print(f"\n--- {row['title']} ({len(apercu.columns)} colonnes) ---")
    print(apercu.columns.tolist())

# 3. Afficher l'aperçu du fichier le plus récent
premier_fichier = next(iter(fichiers_telecharges.values()))
lire_csv_auto(premier_fichier, nrows=5)

               title  size_mb
0  lic-data-2023.csv    256.4
1  lic-data-2022.csv    247.7
2  lic-data-2021.csv    225.8
3  lic-data-2020.csv    240.5
4  lic-data-2019.csv    266.8
5  lic-data-2018.csv    206.3
6  lic-data-2017.csv    222.2
7  lic-data-2016.csv    230.6
Déjà présent : lic-data-2023.csv

--- lic-data-2023.csv (46 colonnes) ---
['Code Commune', 'Commune', 'Code QPV', 'Nom QPV', 'Département', 'Région', 'Statut géo', 'Code', 'Fédération', 'F - 1 à 4 ans', 'F - 5 à 9 ans', 'F - 10 à 14 ans', 'F - 15 à 19 ans', 'F - 20 à 24 ans', 'F - 25 à 29 ans', 'F - 30 à 34 ans', 'F - 35 à 39 ans', 'F - 40 à 44 ans', 'F - 45 à 49 ans', 'F - 50 à 54 ans', 'F - 55 à 59 ans', 'F - 60 à 64 ans', 'F - 65 à 69 ans', 'F - 70 à 74 ans', 'F - 75 à 79 ans', 'F - 80 à 99 ans', 'F - NR', 'H - 1 à 4 ans', 'H - 5 à 9 ans', 'H - 10 à 14 ans', 'H - 15 à 19 ans', 'H - 20 à 24 ans', 'H - 25 à 29 ans', 'H - 30 à 34 ans', 'H - 35 à 39 ans', 'H - 40 à 44 ans', 'H - 45 à 49 ans', 'H - 50 à 54 ans', 'H - 55 à 

,Code Commune,Commune,Code QPV,Nom QPV,Département,Région,Statut géo,Code,Fédération,F - 1 à 4 ans,...,H - 45 à 49 ans,H - 50 à 54 ans,H - 55 à 59 ans,H - 60 à 64 ans,H - 65 à 69 ans,H - 70 à 74 ans,H - 75 à 79 ans,H - 80 à 99 ans,H - NR,Total
0,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,101,FF d'Athlétisme,0,...,0,0,0,0,0,0,0,0,0,6
1,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,103,FF de Badminton,0,...,0,2,1,0,0,0,0,0,0,11
2,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,105,FF de Basketball,0,...,0,0,0,0,0,0,0,0,0,6
3,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,108,FF de Cyclisme,0,...,0,0,0,0,0,0,0,0,0,2
4,1001,L'Abergement-Clémenciat,CSZ,NaN,1,Auvergne-Rhône-Alpes,1.Champ geoc,109,FF d'Équitation,0,...,0,0,0,0,2,0,0,0,0,11


RAJOUT DES BDD DE 2024 ET 2025

In [8]:
import requests
import pandas as pd

# Recherche les jeux de données liés aux licences sur le portail INJEP
url_catalogue = "https://data.sports.gouv.fr/api/explore/v2.1/catalog/datasets"
params = {"where": 'search(dataset_id, "lic")', "limit": 50}

resp = requests.get(url_catalogue, params=params, timeout=30)
resp.raise_for_status()

datasets = [
    {
        "dataset_id": d["dataset_id"],
        "titre": d["metas"]["default"].get("title", ""),
        "modifie_le": d["metas"]["default"].get("modified", ""),
    }
    for d in resp.json()["results"]
]
df_catalogue = pd.DataFrame(datasets)
display(df_catalogue)

,dataset_id,titre,modifie_le
0,lics_histo,Licences annuelles (2002-2025),2025-01-14T16:53:04+00:00
1,lics_dep_age_sexe,Licences annuelles (2025) par département-âge-...,2025-01-10T16:30:14+00:00
2,lics_dep,Licences annuelles (2025) par département,2025-01-10T16:05:26+00:00
3,lics_cc_wide,"Licences annuelles par commune (2023, format l...",2025-01-15T09:09:58+00:00
4,lics_cc,Licences annuelles par commune (2023),2025-01-14T18:07:38+00:00
5,lics_dep_sexe,Licences annuelles par département-sexe (2025),2025-01-16T10:43:39+00:00
6,lics_cc_pyra,Pyramide des âges des licences annuelles par c...,2025-02-19T08:23:04+00:00
7,lics_distr_age,Distribution de l'âge des détenteurs de licenc...,2025-01-16T10:12:31+00:00
8,lics_clubs_dep_ruralurbain,Licences annuelles et établissements sportifs ...,2025-01-15T09:34:06+00:00


In [10]:
import pandas as pd

ids_a_verifier = ["lics_histo", "lics_dep_age_sexe", "lics_dep"]

for ds_id in ids_a_verifier:
    url_csv = f"https://data.sports.gouv.fr/api/explore/v2.1/catalog/datasets/{ds_id}/exports/csv?delimiter=%3B"
    apercu = pd.read_csv(url_csv, sep=";", nrows=5)
    print(f"\n================ {ds_id} ================")
    print("Colonnes :", apercu.columns.tolist())
    display(apercu.head(3))


================ lics_histo ================
Colonnes : ['code_fede_2025', 'code_fede_ref', 'federation', 'annee', 'evol_20ans', 'evol_10ans', 'evol_5ans', 'lics', 'lics_100_2002', 'lics_100_2016', 'fede_gp']


,code_fede_2025,code_fede_ref,federation,annee,evol_20ans,evol_10ans,evol_5ans,lics,lics_100_2002,lics_100_2016,fede_gp
0,101,101,Fédération Française d'Athlétisme,2005-01-01,1.015712,0.183774,0.100433,167007,100.693368,55.304726,1. Olympique
1,101,101,Fédération Française d'Athlétisme,2013-01-01,1.015712,0.183774,0.100433,256459,154.626576,84.926948,1. Olympique
2,101,101,Fédération Française d'Athlétisme,2022-01-01,1.015712,0.183774,0.100433,302583,182.436074,100.201009,1. Olympique



================ lics_dep_age_sexe ================
Colonnes : ['code_fede_2025', 'code_fede_ref', 'sexe', 'trage_full', 'trage', 'dep', 'dep_l', 'newreg', 'newreg_l', 'lics', 'pop', 'lics_pop', 'rank_fede_dep', 'lics_pop_fr', 'is', 'fede_gp']


,code_fede_2025,code_fede_ref,sexe,trage_full,trage,dep,dep_l,newreg,newreg_l,lics,pop,lics_pop,rank_fede_dep,lics_pop_fr,is,fede_gp
0,243,243,F,g - de 30 à 34 ans,30 à 34 ans,43,43 - Haute-Loire,84,84 - Auvergne-Rhône-Alpes,0,5561,0.0,NaN,0.015479,0.0,2. Délégataire non-olympique
1,243,243,F,g - de 30 à 34 ans,30 à 34 ans,47,47 - Lot-et-Garonne,75,75 - Nouvelle-Aquitaine,0,8608,0.0,NaN,0.015479,0.0,2. Délégataire non-olympique
2,243,243,F,g - de 30 à 34 ans,30 à 34 ans,50,50 - Manche,28,28 - Normandie,0,12639,0.0,NaN,0.015479,0.0,2. Délégataire non-olympique



================ lics_dep ================
Colonnes : ['code_fede_2025', 'code_fede_ref', 'rank_fede_dep', 'newreg', 'newreg_l', 'dep', 'dep_l', 'lics', 'pop', 'lics_pop', 'lics_pop_fr', 'is', 'dep_name', 'reg_name', 'federations_sportives', 'federation_court', 'annee']


,code_fede_2025,code_fede_ref,rank_fede_dep,newreg,newreg_l,dep,dep_l,lics,pop,lics_pop,lics_pop_fr,is,dep_name,reg_name,federations_sportives,federation_court,annee
0,271,271,NaN,1.0,01 - Guadeloupe,971,971 - Guadeloupe,0,383187.0,0.0,1.553651,0.0,Guadeloupe,Guadeloupe,Fédération Française des Clubs Alpins et de Mo...,FF des Clubs Alpins et de Montagne,2025
1,271,271,NaN,NaN,COM - COM,975,975 - St-Pierre-et-Miquelon,0,NaN,NaN,1.553651,NaN,St-Pierre-et-Miquelon,COM,Fédération Française des Clubs Alpins et de Mo...,FF des Clubs Alpins et de Montagne,2025
2,271,271,NaN,980.0,980 - Monaco,980,980 - Monaco,24,NaN,NaN,1.553651,NaN,Monaco,Monaco,Fédération Française des Clubs Alpins et de Mo...,FF des Clubs Alpins et de Montagne,2025


NameError: name 'df_licences' is not defined